# Day 2 · Notebook 03 — ONNX export, ONNX Runtime & INT8 for deployment

**Why this notebook matters for Day 3:** our Flask API will *not* import PyTorch. It loads an `.onnx` file
with **ONNX Runtime** (`onnxruntime` ≈ 15 MB wheel, CPU-only) instead of `torch` (≈ 200 MB CPU wheel, >2 GB
with CUDA). Result: a small Docker image, fast cold start, fewer security patches, and a model that any
language/runtime (C++, C#, Java, JavaScript, mobile) can load.

Steps:

1. Export the fine-tuned FP32 model to ONNX (opset 17, input `input`, output `logits`, dynamic batch).
2. Validate: `onnx.checker` + numerical comparison PyTorch vs ONNX Runtime.
3. Look at ONNX Runtime **graph optimisations** (operator fusion).
4. **Static INT8 quantization** with `onnxruntime.quantization` (QDQ, per-channel, calibrated).
5. Benchmark FP32 vs INT8 (size, latency, accuracy) and write `model_card.json` — the contract for Day 3.

Runtime ≈ 3–5 min. Requires `../models/mobilenetv2_cifar10_fp32.pth`.

## 0. Setup

In [ ]:
QUICK_RUN = False

import json, os, time, datetime as dt
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import onnx
import onnxruntime as ort
from torch.utils.data import Subset
from torchvision import datasets, transforms
from torchvision.models import mobilenet_v2

DATA_DIR, MODELS_DIR = Path("..") / "data", Path("..") / "models"
OUT_DIR = Path("outputs"); OUT_DIR.mkdir(exist_ok=True)
IMG_SIZE, SEED = 96, 42
TEST_SUBSET = 500 if QUICK_RUN else 2000
CALIB_SIZE = 64 if QUICK_RUN else 320
BENCH_THREADS = 4
LABELS = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
EXPORT_DIR = OUT_DIR if QUICK_RUN else MODELS_DIR     # smoke tests never touch the shared models/
FP32_ONNX, INT8_ONNX = EXPORT_DIR / "model_fp32.onnx", EXPORT_DIR / "model_int8.onnx"
print("torch", torch.__version__, "| onnx", onnx.__version__, "| onnxruntime", ort.__version__)
print("ORT providers available:", ort.get_available_providers())

`CPUExecutionProvider` is always there. With `onnxruntime-gpu` you would also see `CUDAExecutionProvider`
(and optionally `TensorrtExecutionProvider`) — same model file, different *execution provider*.

## 1. Load the model and the test data

In [ ]:
model = mobilenet_v2(weights=None)
model.classifier[1] = nn.Linear(1280, 10)
model.load_state_dict(torch.load(MODELS_DIR / "mobilenetv2_cifar10_fp32.pth", map_location="cpu",
                                 weights_only=True))
model.eval()

test_tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(),
                              transforms.Normalize(MEAN, STD)])     # the Day-3 contract
def fixed_subset(n_total, n, seed):
    g = torch.Generator().manual_seed(seed)
    return torch.randperm(n_total, generator=g)[:n].tolist()

test_full = datasets.CIFAR10(DATA_DIR, train=False, download=True, transform=test_tf)
test_ds = Subset(test_full, fixed_subset(len(test_full), TEST_SUBSET, SEED))
X_test = np.stack([test_ds[i][0].numpy() for i in range(len(test_ds))]).astype(np.float32)
y_test = np.array([test_ds[i][1] for i in range(len(test_ds))])
print("X_test", X_test.shape, X_test.dtype, "| y_test", y_test.shape)

From here on the test set is a plain **NumPy array** — exactly what ONNX Runtime consumes (no tensors).

## 2. Export to ONNX

`torch.onnx.export` runs the model once on a dummy input and records the graph of operations.

* **`input_names` / `output_names`** — the names clients use; the Day-3 API calls
  `session.run(["logits"], {"input": x})`.
* **`dynamic_axes`** — mark dim 0 as symbolic (`"batch"`), otherwise the graph is frozen to batch = 1.
* **`opset_version=17`** — the ONNX operator-set version; the runtime must support it (ORT does up to 21+).
* **`dynamo=False`** — since PyTorch 2.9 the default exporter is the new `torch.export`-based
  ("dynamo") exporter, which expresses dynamic dims via `dynamic_shapes=`. We use the mature *TorchScript-based*
  exporter here because it supports `dynamic_axes` directly and produces a single file for this model.
  Both produce standard ONNX; the dynamo exporter is the future for complex models (control flow, LLMs).
* The model must be in **`eval()`** mode: Dropout disappears and BatchNorm uses running statistics (and is
  folded into the conv weights by constant folding).

In [ ]:
import warnings   # we choose the legacy exporter on purpose (see above) -> hide its deprecation notice
warnings.filterwarnings("ignore", message=r".*legacy TorchScript-based ONNX export.*")

dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
torch.onnx.export(
    model, dummy, str(FP32_ONNX),
    input_names=["input"], output_names=["logits"],
    dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=17,
    dynamo=False,
)
onnx_model = onnx.load(str(FP32_ONNX))
onnx.checker.check_model(onnx_model)                        # structural validation
# make sure the file is self-contained (no side-car .data file with external weights)
onnx.save(onnx_model, str(FP32_ONNX), save_as_external_data=False)
print("exported:", FP32_ONNX, f"{FP32_ONNX.stat().st_size/1e6:.2f} MB")
print("external data file present:", Path(str(FP32_ONNX) + ".data").exists())

Weights > 2 GB must be stored as *external data* (protobuf limit). Our 9 MB model fits in one file — one
artefact to copy into the Docker image.

### What is inside the graph?

In [ ]:
def describe(vi):
    dims = [d.dim_param or d.dim_value for d in vi.type.tensor_type.shape.dim]
    return f"{vi.name}: {onnx.TensorProto.DataType.Name(vi.type.tensor_type.elem_type)} {dims}"

print("opset :", [(o.domain or "ai.onnx", o.version) for o in onnx_model.opset_import])
print("inputs :", [describe(i) for i in onnx_model.graph.input])
print("outputs:", [describe(o) for o in onnx_model.graph.output])
ops = Counter(n.op_type for n in onnx_model.graph.node)
print("nodes  :", sum(ops.values()), dict(ops))
print("contains BatchNormalization nodes:", "BatchNormalization" in ops)

* Input is `FLOAT ['batch', 3, 96, 96]` — batch is symbolic ✔.
* There are **no BatchNormalization nodes**: in eval mode BN is an affine transform and was *folded into the
  preceding Conv weights* during export (constant folding) — a first graph optimisation for free.
* `Clip` = ReLU6 (`clip(x, 0, 6)`), the `Constant` nodes hold its bounds 0 and 6, `Add` = the residual
  connections, `Gemm` = the final Linear layer. The conv weights are stored as *initializers* (not nodes).

## 3. Validate numerically: PyTorch vs ONNX Runtime

Passing the checker only means the file is well-formed. We must also check that it computes **the same
function**. Small differences (~1e-5) are expected: different kernels, different summation order.

In [ ]:
sess = ort.InferenceSession(str(FP32_ONNX), providers=["CPUExecutionProvider"])
print("session inputs :", [(i.name, i.shape, i.type) for i in sess.get_inputs()])
print("session outputs:", [(o.name, o.shape, o.type) for o in sess.get_outputs()])

for bs in [1, 7, 32]:                                       # dynamic batch works for any size
    xb = X_test[:bs]
    with torch.no_grad():
        ref = model(torch.from_numpy(xb)).numpy()
    out = sess.run(["logits"], {"input": xb})[0]
    print(f"batch {bs:2d}: output {out.shape} | max|diff| = {np.abs(ref-out).max():.2e} | "
          f"allclose(atol=1e-4): {np.allclose(ref, out, atol=1e-4)} | "
          f"same top-1: {(ref.argmax(1) == out.argmax(1)).all()}")

✔ Same outputs up to ~1e-5 for several batch sizes. Always add such a test to your CI when you export models.

## 4. ONNX Runtime graph optimisations (operator fusion)

When a session is created ORT rewrites the graph. Levels:

* `ORT_DISABLE_ALL` — run the graph as exported.
* `ORT_ENABLE_BASIC` — constant folding, redundant node elimination, simple fusions.
* `ORT_ENABLE_EXTENDED` — complex fusions (e.g. Conv + activation → `FusedConv`, attention fusions for Transformers).
* `ORT_ENABLE_ALL` (default) — all of the above + layout optimisations (e.g. NCHWc blocked layout for x86 SIMD).

**Why fusion helps:** each separate op writes its output to memory and the next op reads it back. Fusing
`Conv → BN → ReLU` into one kernel computes the activation while the data is still in registers/cache —
fewer memory round-trips and fewer kernel launches. On small models memory traffic, not FLOPs, is often the
bottleneck.

In [ ]:
def ort_session(path, level=ort.GraphOptimizationLevel.ORT_ENABLE_ALL, threads=BENCH_THREADS, save_to=None):
    so = ort.SessionOptions()
    so.graph_optimization_level = level
    so.intra_op_num_threads = threads
    if save_to:
        so.optimized_model_filepath = str(save_to)
    return ort.InferenceSession(str(path), so, providers=["CPUExecutionProvider"])

def ort_latency(sess, x, n=200, warmup=20):
    for _ in range(warmup):
        sess.run(None, {"input": x})
    t = []
    for _ in range(n):
        t0 = time.perf_counter(); sess.run(None, {"input": x}); t.append((time.perf_counter() - t0) * 1000)
    return float(np.median(t))

x1 = X_test[:1]
rows = []
for name, level in [("DISABLE_ALL", ort.GraphOptimizationLevel.ORT_DISABLE_ALL),
                    ("ENABLE_BASIC", ort.GraphOptimizationLevel.ORT_ENABLE_BASIC),
                    ("ENABLE_EXTENDED", ort.GraphOptimizationLevel.ORT_ENABLE_EXTENDED),
                    ("ENABLE_ALL", ort.GraphOptimizationLevel.ORT_ENABLE_ALL)]:
    opt_path = OUT_DIR / f"fp32_opt_{name}.onnx"
    s = ort_session(FP32_ONNX, level, save_to=opt_path)
    g = onnx.load(str(opt_path))
    rows.append({"level": name, "nodes": len(g.graph.node),
                 "op types": dict(Counter(n.op_type for n in g.graph.node).most_common(4)),
                 "latency_ms": round(ort_latency(s, x1), 3)})
display(pd.DataFrame(rows))

Read the table from top to bottom (the `Constant` nodes are already gone after loading):

* `ENABLE_EXTENDED`: every `Conv` followed by `Clip` becomes a single **`FusedConv`** node (activation computed
  inside the convolution kernel) → far fewer nodes.
* `ENABLE_ALL`: the convolutions are rewritten into the x86-optimised **NCHWc** blocked memory layout
  (`Conv` ops of the `com.microsoft.nchwc` domain with the activation still fused, plus a few
  `ReorderInput`/`ReorderOutput` layout conversions) — usually the biggest latency win on x86.

On a busy laptop small differences can be within noise, but the trend is clear. `ENABLE_ALL` is the default — you
get all of this for free just by using ONNX Runtime.

> Tip for production: the optimised graph can be saved once (`optimized_model_filepath`) to skip the
> optimisation step at start-up — but the saved `ENABLE_ALL` graph is hardware-specific. We ship the
> portable model and let ORT optimise on the target machine.

### PyTorch eager vs ONNX Runtime (same model, same threads, batch = 1)

In [ ]:
torch.set_num_threads(BENCH_THREADS)
xt = torch.from_numpy(x1)
with torch.inference_mode():
    for _ in range(20):
        model(xt)
    tt = []
    for _ in range(200):
        t0 = time.perf_counter(); model(xt); tt.append((time.perf_counter() - t0) * 1000)
torch_lat = float(np.median(tt))
ort_fp32_lat = ort_latency(ort_session(FP32_ONNX), x1)
print(f"PyTorch eager FP32 : {torch_lat:.2f} ms")
print(f"ONNX Runtime FP32  : {ort_fp32_lat:.2f} ms   (x{torch_lat/ort_fp32_lat:.1f} faster)")

For small models at batch = 1, PyTorch eager spends much of its time in Python/dispatcher overhead per
operator; ONNX Runtime executes a pre-planned, fused graph in C++. This alone often gives a multi-× speed-up
before any quantization.

## 5. Static INT8 quantization with ONNX Runtime

Same idea as PyTorch static PTQ, different tool:

1. **Pre-process** (`quant_pre_process`): symbolic shape inference + graph clean-up so the quantizer knows
   every tensor's shape.
2. **Calibration data reader**: a small class whose `get_next()` returns `{"input": batch}` dicts and `None`
   when exhausted — a few hundred **training** images with the **inference** preprocessing.
3. **`quantize_static`** with
   * `QuantFormat.QDQ` — inserts `QuantizeLinear`/`DequantizeLinear` pairs around ops. Portable (the same
     representation TensorRT and other runtimes understand); ORT fuses `DQ → Conv → Q` into an INT8 kernel.
   * `per_channel=True` — one scale per output channel for weights (as seen in Notebook 02).
   * activations `QUInt8`, weights `QInt8` ("U8S8") — the recommended combination on x86 CPUs.
   * `CalibrationMethod.MinMax` — simplest; `Entropy` and `Percentile` clip outliers (try them in the lab).

In [ ]:
from onnxruntime.quantization import (CalibrationDataReader, CalibrationMethod, QuantFormat,
                                      QuantType, quantize_static)
from onnxruntime.quantization.shape_inference import quant_pre_process

train_plain = datasets.CIFAR10(DATA_DIR, train=True, download=True, transform=test_tf)
calib_idx = fixed_subset(len(train_plain), CALIB_SIZE, SEED + 1)
X_calib = np.stack([train_plain[i][0].numpy() for i in calib_idx]).astype(np.float32)

class CifarCalibrationReader(CalibrationDataReader):
    def __init__(self, data, batch_size=32):
        self.batches = iter([{"input": data[i:i + batch_size]} for i in range(0, len(data), batch_size)])
    def get_next(self):
        return next(self.batches, None)          # None signals "no more data"

prep_path = OUT_DIR / "model_fp32_prep.onnx"
quant_pre_process(str(FP32_ONNX), str(prep_path))
t0 = time.time()
quantize_static(
    model_input=str(prep_path),
    model_output=str(INT8_ONNX),
    calibration_data_reader=CifarCalibrationReader(X_calib),
    quant_format=QuantFormat.QDQ,
    per_channel=True,
    activation_type=QuantType.QUInt8,
    weight_type=QuantType.QInt8,
    calibrate_method=CalibrationMethod.MinMax,
)
int8_model = onnx.load(str(INT8_ONNX))
onnx.checker.check_model(int8_model)
onnx.save(int8_model, str(INT8_ONNX), save_as_external_data=False)
print(f"quantized with {len(X_calib)} calibration images in {time.time()-t0:.1f}s")
print(f"FP32: {FP32_ONNX.stat().st_size/1e6:.2f} MB -> INT8: {INT8_ONNX.stat().st_size/1e6:.2f} MB")
ops8 = Counter(n.op_type for n in int8_model.graph.node)
print("INT8 graph op types:", dict(ops8.most_common(6)))

The QDQ graph still contains the float ops (`Conv`, `Gemm`, …) but their inputs come through
`DequantizeLinear` from INT8 weights, and their outputs go through `QuantizeLinear`. At session creation ORT
recognises the `DQ → Conv → Q` patterns and replaces them with integer kernels (`QLinearConv`). Weights are
stored as INT8 → ~3.5× smaller file.

## 6. Benchmark FP32 vs INT8

In [ ]:
def evaluate_onnx(path, batch=100):
    s = ort_session(path)
    preds = np.concatenate([s.run(["logits"], {"input": X_test[i:i + batch]})[0].argmax(1)
                            for i in range(0, len(X_test), batch)])
    return preds, (preds == y_test).mean(), ort_latency(s, x1)

bench = {}
for name, path in [("fp32", FP32_ONNX), ("int8", INT8_ONNX)]:
    preds, acc, lat = evaluate_onnx(path)
    bench[name] = {"file": path.name, "size_mb": round(path.stat().st_size / 1e6, 3),
                   "accuracy": round(float(acc), 4), "latency_ms": round(lat, 3), "preds": preds}

agree = (bench["fp32"]["preds"] == bench["int8"]["preds"]).mean()
table = pd.DataFrame({k: {c: v for c, v in d.items() if c != "preds"} for k, d in bench.items()}).T
display(table)
f, i = bench["fp32"], bench["int8"]
print(f"INT8 vs FP32: size x{f['size_mb']/i['size_mb']:.2f} smaller | speed-up x{f['latency_ms']/i['latency_ms']:.2f}"
      f" | accuracy {100*(i['accuracy']-f['accuracy']):+.2f} pp | top-1 agreement {100*agree:.1f} %")

**Top-1 agreement** (how often INT8 and FP32 predict the same class) is a useful extra metric: it tells you
how many *user-visible* answers change after quantization, independent of labels.

Batch-1 latency of such a small model is only ~1 ms either way; INT8 gains are larger with bigger models,
bigger batches, and on CPUs with VNNI/AMX instructions. The **size** reduction is always there (smaller
container, faster download to edge devices).

### Why not dynamic quantization for the ONNX model?

`onnxruntime.quantization.quantize_dynamic` needs no calibration, but for convolutions it produces
`ConvInteger` ops that compute activation scales at run time — usually **slower** than FP32 for CNNs. We use
it only as a fallback if static quantization fails (e.g. no representative data available). Quick check:

In [ ]:
from onnxruntime.quantization import quantize_dynamic
dyn_path = OUT_DIR / "model_dynamic_int8.onnx"
quantize_dynamic(str(prep_path), str(dyn_path), weight_type=QuantType.QUInt8)
_, dyn_acc, dyn_lat = evaluate_onnx(dyn_path)
print(f"dynamic INT8: size {dyn_path.stat().st_size/1e6:.2f} MB | accuracy {dyn_acc:.4f} | latency {dyn_lat:.2f} ms")

Compare with the static INT8 row above: similar size, but typically worse latency (and sometimes accuracy)
— for CNNs static quantization is the right choice.

## 7. Write the model card (the contract for Day 3)

The Flask API reads `model_card.json` to know the input size, normalisation, labels, tensor names and which
file to load. Keeping this metadata **next to the model** (instead of hard-coding it in the server) means we can
retrain or swap the model without touching the API code.

In [ ]:
card = {
    "img_size": IMG_SIZE,
    "mean": MEAN,
    "std": STD,
    "labels": LABELS,
    "input_name": "input",
    "output_name": "logits",
    "variants": {k: {c: v for c, v in d.items() if c != "preds"} for k, d in bench.items()},
    "trained_on": "CIFAR-10 subset",
    "created": dt.datetime.now().isoformat(timespec="seconds"),
}
if QUICK_RUN:
    print("QUICK_RUN: model card not written (smoke test)")
else:
    (MODELS_DIR / "model_card.json").write_text(json.dumps(card, indent=2))
    (MODELS_DIR / "labels.json").write_text(json.dumps(LABELS, indent=2))
    print("written:", MODELS_DIR / "model_card.json")
print(json.dumps(card, indent=2))

## 8. Day-3 preview: inference **without PyTorch**

The server will only have `numpy`, `Pillow` and `onnxruntime`. We re-implement the preprocessing in NumPy and
check it matches the torchvision pipeline bit-for-bit — a classic source of silent accuracy loss in production
("training/serving skew").

In [ ]:
from PIL import Image
card = json.loads((MODELS_DIR / "model_card.json").read_text()) if not QUICK_RUN else card

def preprocess_numpy(img: Image.Image, card) -> np.ndarray:
    img = img.convert("RGB").resize((card["img_size"], card["img_size"]), Image.BILINEAR)
    x = np.asarray(img, dtype=np.float32) / 255.0                      # HWC in [0,1]  (== ToTensor)
    x = (x - np.array(card["mean"], np.float32)) / np.array(card["std"], np.float32)
    return x.transpose(2, 0, 1)[None].astype(np.float32)               # -> NCHW, batch of 1

raw = datasets.CIFAR10(DATA_DIR, train=False, download=False)          # PIL images, no transform
pil_img, label = raw[test_ds.indices[0]]
x_np = preprocess_numpy(pil_img, card)
x_tv = test_tf(pil_img).unsqueeze(0).numpy()
print("max |numpy - torchvision| =", float(np.abs(x_np - x_tv).max()))

def softmax(z):
    e = np.exp(z - z.max(1, keepdims=True)); return e / e.sum(1, keepdims=True)

for variant in ["fp32", "int8"]:
    s = ort.InferenceSession(str(EXPORT_DIR / card["variants"][variant]["file"]), providers=["CPUExecutionProvider"])
    probs = softmax(s.run([card["output_name"]], {card["input_name"]: x_np})[0])[0]
    top = probs.argsort()[::-1][:3]
    print(f"{variant}: true={LABELS[label]:>10s} | top-3:",
          ", ".join(f"{card['labels'][k]} {probs[k]:.3f}" for k in top))
plt.figure(figsize=(2, 2)); plt.imshow(pil_img); plt.axis("off"); plt.title(LABELS[label]); plt.show()

The difference is at float rounding level (~1e-7) → the server's NumPy preprocessing is equivalent to training.
This small function is essentially the core of tomorrow's `/predict` endpoint.

## 9. The full picture — PyTorch techniques (Notebook 02) + ONNX Runtime

In [ ]:
rows = []
csv = OUT_DIR / "day2_torch_optimization_results.csv"
if csv.exists():
    tdf = pd.read_csv(csv)
    for _, r in tdf.iterrows():
        rows.append({"technique": "PyTorch: " + r["technique"], "size_mb": r["size_mb"],
                     "latency_ms": r["latency_ms"], "accuracy": r["accuracy"]})
else:
    print("(run Notebook 02 first to include the PyTorch results)")
rows += [{"technique": "ONNX Runtime FP32", "size_mb": bench["fp32"]["size_mb"],
          "latency_ms": bench["fp32"]["latency_ms"], "accuracy": bench["fp32"]["accuracy"]},
         {"technique": "ONNX Runtime INT8 (static QDQ)", "size_mb": bench["int8"]["size_mb"],
          "latency_ms": bench["int8"]["latency_ms"], "accuracy": bench["int8"]["accuracy"]},
         {"technique": "ONNX Runtime INT8 (dynamic)", "size_mb": round(dyn_path.stat().st_size / 1e6, 3),
          "latency_ms": round(dyn_lat, 3), "accuracy": round(float(dyn_acc), 4)}]
full = pd.DataFrame(rows)
display(full)
full.to_csv(OUT_DIR / "day2_all_results.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(full["latency_ms"], full["accuracy"], s=full["size_mb"] * 40, alpha=.6)
for _, r in full.iterrows():
    ax.annotate(r["technique"].replace("PyTorch: ", "PT: "), (r["latency_ms"], r["accuracy"]),
                fontsize=7, xytext=(4, 2), textcoords="offset points")
ax.set_xscale("log"); ax.set_xlabel(f"latency, batch=1, {BENCH_THREADS} threads [ms] (log)")
ax.set_ylabel("accuracy"); ax.set_title("Accuracy vs latency (bubble area ∝ size on disk)")
ax.grid(alpha=.3); plt.show()

The winners for serving on a CPU sit in the **top-left with small bubbles**: ONNX Runtime FP32/INT8.
That's why Day 3 serves `model_int8.onnx` (with `model_fp32.onnx` as the reference variant) from a slim
container with no PyTorch inside.

## Summary

* Export with explicit **names, dynamic batch and opset**; validate with `onnx.checker` **and** a numerical
  comparison.
* ONNX Runtime gives graph optimisations (BN folding, Conv+activation fusion, layout transforms) for free.
* **Static INT8 (QDQ, per-channel, calibrated)** → ~3.5× smaller with a small accuracy change; dynamic
  quantization is not a good fit for CNNs.
* Keep preprocessing & metadata in a **model card** next to the model; test the NumPy preprocessing against the
  training pipeline.

➡️ Continue with **04_student_lab.ipynb**.